[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/math/reflections/six_hundred_cell.ipynb)

# The 600-Cell from the Icosahedron's Mirrors

This example constructs the 600-cell, the regular four-dimensional polytope bounded by 600 tetrahedral facets across 120 vertices, directly from the three mirrors of the icosahedron. Every rotation of the icosahedron lifts to two opposite unit spinors, and because a three-dimensional unit spinor has four components of unit norm, those 120 spinors sit on the three-sphere as the vertices of the 600-cell, each surrounded by twelve nearest neighbours.

Each mirror is modelled as an extrinsic unit plane `Plane`, so pairwise products `second * first`, `third * first` and `third * second` form the generating rotors around the icosahedron's twofold, threefold and fivefold axes. Multiplying batches of products across all generators via `spinors[:, None] * step_factors[None, :]` closes into the 120-element binary icosahedral group, and left-and-right multiplication `left * spinors * right` rotates the three-sphere in two orthogonal four-dimensional planes at once.

In Hamilton quaternion and $4 \times 4$ $\mathrm{SO}(4)$ matrix notation, the 120 unit icosians split in an aligned coordinate frame into the eight basis units $\pm 1, \pm i, \pm j, \pm k$, the sixteen half-integer combinations $\tfrac12(\pm 1 \pm i \pm j \pm k)$, and the 96 even-permutation coordinates $\tfrac12(\pm\varphi \pm 1\,i \pm \varphi^{-1} j + 0\,k)$ with golden ratio $\varphi$, while double rotations act on column 4-vectors through paired left and right real isoclinic $4 \times 4$ matrices $M_L(q_L)\,M_R(q_R)$.

In [ ]:
# In Colab, install numga and its examples from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline

from collections.abc import Iterator

import numpy as np

from numga import NumpyContext, concatenate, stack
from numga.algebras import VGA3D
from examples.util.animation import show_states
from examples.math.reflections import render

ga = VGA3D
mv = NumpyContext(ga).multivector
Rotor = ga.gatype.rotor()                                                       # Rotor+

## Mirror planes and generating rotors

Three outward unit mirror vectors whose planes meet at a half turn divided by two, three and five determine one another's inner products as minus the cosines of those dihedral angles. Resolving the third mirror into its horizontal reciprocal-basis part plus its vertical unit-length complement yields all three mirrors, whose pairwise geometric products generate the rotational group.

In [ ]:
# Dihedral angles between pairs of icosahedral mirrors.
first_second = np.pi / 2
first_third = np.pi / 3
second_third = np.pi / 5

In [ ]:
first = mv.x                                                                    # [] Plane+
first_turn = (mv.xy * (-(np.pi - first_second) / 2)).exp()                      # [] Rotor+
second = first_turn >> first                                                    # [] Plane+
plane = first ^ second                                                          # [] Bivector+
in_plane = -np.cos(first_third) * (second | plane.inverse()) - np.cos(second_third) * (plane.inverse() | first)   # [] Plane+
normal = mv.z                                                                   # [] Plane+
height = (1 - in_plane.squared()).square_root()                                 # [] Scalar+
third = in_plane + height * normal                                              # [] Plane+
generating_rotors = stack([second * first, third * first, third * second])      # [3] Rotor+

## Closing the binary icosahedral group

Appending the scalar identity to the three generating rotors and multiplying eight factors in every combination reaches every element of the binary icosahedral group, covering all 120 distinct unit spinors along multiple word paths.

In [ ]:
# Word length sufficient to reach all 120 elements from the three generating rotors.
n_factors = 8

In [ ]:
step_factors = concatenate([mv.scalar([[1]]), generating_rotors])               # [4] Rotor+
spinors = step_factors                                                          # [...] Rotor+
for _ in range(n_factors - 1):
    spinors = (spinors[:, None] * step_factors[None, :]).reshape((-1,))         # [...] Rotor+

## Double rotation of the 600-cell

Multiplying every spinor on the left in one coordinate bivector plane and on the right in an orthogonal plane at the same rate performs an isoclinic rotation of the three-sphere. Over a half-angle sweep to a quarter turn on each side, the left and right factors reach group elements themselves, returning the 600-cell seamlessly to its starting vertex set.

In [ ]:
# Frame count and frame duration for one closed loop.
n_frames = 150
frame_duration_ms = 40

In [ ]:
def spinor_frames(n_frames: int) -> Iterator[Rotor]:
    """Yields the spinor batch under simultaneous left `xy` and right `yz` rotors across `n_frames` steps."""
    for angle in np.linspace(0.0, np.pi, n_frames, endpoint=False):
        left = (mv.xy * (angle / 2)).exp()                                      # [] Rotor+
        right = (mv.yz * (angle / 2)).exp()                                     # [] Rotor+
        yield left * spinors * right


recorded_spinors = show_states(
    spinor_frames(n_frames), render.polytope_animation,
    name="six_hundred_cell", duration_ms=frame_duration_ms, n_frames=n_frames,
);

Perspective projection from beyond the scalar pole sends the 120 spinors to blue vertex spheres and joins each vertex by gold struts to its twelve nearest neighbours, turning nested shells through one another as the four-dimensional polytope rotates.

In [ ]:
# checks
vertices = render.distinct_spinors(recorded_spinors[0])                         # [120] Rotor+
assert len(vertices) == 120
assert ((vertices.norm() - 1).abs() < 1e-10).all()
products = render.distinct_spinors((vertices[:, None] * vertices[None, :]).reshape((-1,)))   # [120] Rotor+
assert len(products) == 120
pairwise_distances = (products[:, None] - vertices[None, :]).norm()             # [120, 120] Scalar+
assert (pairwise_distances.min(axis=-1) < 1e-8).all()
chord_lengths = (vertices[:, None] - vertices[None, :]).norm()                  # [120, 120] Scalar+
golden_ratio = (1 + np.sqrt(5)) / 2
assert (((chord_lengths - 1 / golden_ratio).abs() < 1e-8).sum(axis=-1) == 12).all()